# Example queries: `mapped_column` (resstock_oedi)

Auto-generated from `tests/query_snapshots/mapped_column.json`. Each cell
runs one entry from the snapshot suite. Regenerate by running the
matching test with `--update-snapshot` or `--overwrite-snapshot`.


In [ ]:
from pathlib import Path
from buildstock_query import BuildStockQuery
import pandas as pd


## Construct the BuildStockQuery object

`cache_folder` points at the snapshot test cache directory so this
notebook reuses parquets that the test suite has already downloaded
from Athena. Queries that are already cached return immediately;
anything new still hits Athena.


In [ ]:
# Cache folder: resolve `tests/query_snapshots/resstock_oedi_cache` regardless
# of which directory Jupyter was launched from.
_CANDIDATES = [
    Path.cwd() / "tests/query_snapshots/resstock_oedi_cache",            # repo root
    Path.cwd() / "../query_snapshots/resstock_oedi_cache",               # tests/example_notebooks/
    Path.cwd() / "query_snapshots/resstock_oedi_cache",                  # tests/
]
_CACHE = next((p.resolve() for p in _CANDIDATES if p.exists()), _CANDIDATES[0].resolve())
bsq = BuildStockQuery(
    "rescore",
    "buildstock_sdr",
    "resstock_2024_amy2018_release_2",
    buildstock_type="resstock",
    db_schema="resstock_oedi_vu",
    skip_reports=True,
    cache_folder=str(_CACHE),
)


## `mapped_column_simple_bldg_type_group_by`

Annual electricity grouped by a MappedColumn that collapses building types into broader categories, CO only.


In [ ]:
result = bsq.query(
    enduses=['out.electricity.total.energy_consumption'],
    restrict=[('state', ['CO'])],
    group_by=[MappedColumn(bsq=<buildstock_query.main.BuildStockQuery object at 0x152f440b0>, name='simple_bldg_type', mapping_dict={'Mobile Home': 'MH', 'Single-Family Detached': 'SF', 'Single-Family Attached': 'SF', 'Multi-Family with 2 - 4 Units': 'MF', 'Multi-Family with 5+ Units': 'MF'}, key=Column('in.geometry_building_type_recs', String(), table=<bs>))],
)
result.head() if hasattr(result, 'head') else result


# shape: (3, 4)
simple_bldg_type  sample_count  units_count  electricity.total.energy_consumption
              MF          2470 6.231850e+05                          4.104092e+09
              MH           391 9.864994e+04                          7.898215e+08
              SF          6564 1.656108e+06                          1.769998e+10


## `mapped_column_load_factor_enduse`

MappedColumn used as an enduse rather than a group_by column, CO only. Mirrors the advanced_usage notebook pattern where a numeric mapping (per-building-type load factor) is summed across the population. Exercises the enduse path through MappedColumn.


In [ ]:
result = bsq.query(
    enduses=[MappedColumn(bsq=<buildstock_query.main.BuildStockQuery object at 0x152f440b0>, name='bldg_load_factor', mapping_dict={'Mobile Home': 0.5, 'Single-Family Detached': 1.0, 'Single-Family Attached': 0.8, 'Multi-Family with 2 - 4 Units': 0.6, 'Multi-Family with 5+ Units': 0.4}, key=Column('in.geometry_building_type_recs', String(), table=<bs>))],
    group_by=['geometry_building_type_recs'],
    restrict=[('state', ['CO'])],
)
result.head() if hasattr(result, 'head') else result


# shape: (5, 4)
  geometry_building_type_recs  sample_count  units_count  bldg_load_factor
                  Mobile Home           391 9.864994e+04      4.932497e+04
Multi-Family with 2 - 4 Units           469 1.183295e+05      7.099768e+04
   Multi-Family with 5+ Units          2001 5.048556e+05      2.019422e+05
       Single-Family Attached           664 1.675283e+05      1.340226e+05
       Single-Family Detached          5900 1.488580e+06      1.488580e+06
